# Modelagem supervisionada: pré-processamento e baseline (Fase 3)

**Objetivo:** montar o pipeline de pré-processamento e treinar os modelos de referência
(baselines) para prever se um aluno **não** será alfabetizado (classe de risco).

**Entradas:**
- `data/sample/features_aluno`: a camada Gold (1 linha por aluno avaliado em 2024).
- `src/config/features_modelo.json`: as features e o tratamento de cada uma, definidos
  no notebook `02_selecao_features`. Nenhuma lista de features é digitada aqui.

## 1. Setup e carga


In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import warnings
from sklearn.exceptions import UndefinedMetricWarning
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

pd.set_option("display.max_columns", 50)

RANDOM_STATE = 42   # semente fixa: garante que os sorteios se repitam em qualquer execução

CAMINHO_GOLD = "../data/sample/features_aluno"
CAMINHO_DECISOES = Path("../src/config/features_modelo.json")

df = pd.read_parquet(CAMINHO_GOLD)
decisoes = json.loads(CAMINHO_DECISOES.read_text(encoding="utf-8"))

NUMERICAS_LOG = decisoes["numericas_log"]
NUMERICAS_PADRAO = decisoes["numericas_padrao"]
CATEGORICAS = decisoes["categoricas"]
BINARIAS = decisoes["binarias"]
FEATURES = NUMERICAS_LOG + NUMERICAS_PADRAO + CATEGORICAS + BINARIAS

print(f"Gold: {df.shape} | features lidas do JSON: {len(FEATURES)}")

Gold: (1610754, 35) | features lidas do JSON: 24


## 2. Features (X), alvo (y) e grupos

- **X:** as 24 features. As binárias são convertidas de verdadeiro/falso/vazio para
  1/0/vazio, porque o `SimpleImputer` não aceita valores lógicos com vazios
  (seção 8 do notebook 02).
- **y:** o alvo invertido, **1 = não alfabetizado (risco)** (seção 3 do notebook 02).
- **grupos:** o município de cada aluno, usado para que treino e teste nunca
  compartilhem o mesmo município.

In [2]:
X = df[FEATURES].copy()
for coluna in BINARIAS:
    X[coluna] = X[coluna].astype(float)   # True/False/vazio -> 1.0/0.0/NaN

y = (~df[decisoes["alvo"]]).astype(int).rename("nao_alfabetizado")
grupos = df[decisoes["coluna_grupo"]]

print(f"X: {X.shape}")
print(f"Taxa de risco (y = 1): {y.mean():.1%}")
print(f"Municípios: {grupos.nunique()}")

X: (1610754, 24)
Taxa de risco (y = 1): 40.6%
Municípios: 5450


## 3. Divisão entre treino e teste

A divisão é feita **por município**: 20% dos municípios vão para o teste, com todos os
seus alunos. Assim, nenhum município aparece nos dois conjuntos.

**Por que por município:** as features são quase todas municipais e se repetem em todos
os alunos de cada município (pseudo-replicação, `01_eda`, seção 4). Numa divisão
aleatória por aluno, o modelo veria no teste municípios que já conheceu no treino, e a
métrica mediria a capacidade de reconhecer municípios, não de generalizar. A divisão
por município responde à pergunta de negócio: o modelo acerta em municípios que
nunca viu?

**O conjunto de teste é reservado para a avaliação final** (etapa de avaliação do
cronograma). Toda comparação de modelos e ajuste de hiperparâmetros usa validação
cruzada **dentro do treino**.

In [3]:
from sklearn.model_selection import GroupShuffleSplit

divisor = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
idx_treino, idx_teste = next(divisor.split(X, y, groups=grupos))

X_treino, X_teste = X.iloc[idx_treino], X.iloc[idx_teste]
y_treino, y_teste = y.iloc[idx_treino], y.iloc[idx_teste]
grupos_treino, grupos_teste = grupos.iloc[idx_treino], grupos.iloc[idx_teste]

resumo_divisao = pd.DataFrame({
    "alunos": [len(y_treino), len(y_teste)],
    "pct_alunos": [len(y_treino) / len(y), len(y_teste) / len(y)],
    "municipios": [grupos_treino.nunique(), grupos_teste.nunique()],
    "taxa_risco": [y_treino.mean(), y_teste.mean()],
}, index=["treino", "teste"]).round(3)
print(resumo_divisao)

# conferência: nenhum município nos dois conjuntos
assert set(grupos_treino).isdisjoint(set(grupos_teste))

         alunos  pct_alunos  municipios  taxa_risco
treino  1320423        0.82        4360       0.409
teste    290331        0.18        1090       0.393


**Resultado:** o treino tem 4.360 municípios (82% dos alunos, 1.320.423) e o teste,
1.090 municípios (18% dos alunos, 290.331), sem nenhum município em comum. A taxa de
risco é de 40,9% no treino e 39,3% no teste, uma diferença de 1,6 ponto percentual.

Como 20% dos municípios não correspondem exatamente a 20% dos alunos (os municípios têm
tamanhos muito diferentes), o percentual de alunos no teste varia. A taxa de risco
próxima nos dois conjuntos mostra que o sorteio não desequilibrou o alvo.

## 4. Pipeline de pré-processamento

Cada grupo de features recebe o tratamento definido no notebook 02:

| Grupo | Features | Imputação | Transformação | Encoding |
|---|---|---|---|---|
| Numéricas com log | 6 | Mediana | `log1p` + padronização | Não se aplica (já são números) |
| Numéricas | 15 | Mediana | Padronização | Não se aplica (já são números) |
| Categóricas | `regiao`, `sigla_uf` | Mais frequente | — | **One-hot**: uma coluna 0/1 por categoria |
| Binária | `mun_elegivel_meta` | Constante 0 | — | Conversão direta: verdadeiro/falso → 1/0 (seção 2) |


As categóricas usam **one-hot** porque não têm ordem natural (um encoding ordinal criaria
uma ordem falsa entre regiões e estados) e têm poucas categorias (5 regiões e 25 UFs),
o que mantém o número de colunas baixo.

**Por que um pipeline:** os valores aprendidos no pré-processamento (as medianas, a média
e o desvio da padronização, as categorias do one-hot) são calculados **somente com os
dados de treino** e depois aplicados ao teste. Se fossem calculados com a base inteira,
informação do teste vazaria para o treino. Dentro do pipeline, essa ordem é garantida
automaticamente, inclusive em cada rodada da validação cruzada.


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

imputacao = decisoes["imputacao"]

def criar_preprocessador():
    """Monta o pré-processamento com as decisões do notebook 02."""
    numericas_log = Pipeline([
        ("imputar", SimpleImputer(strategy=imputacao["numericas"])),
        ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("padronizar", StandardScaler()),
    ])
    numericas = Pipeline([
        ("imputar", SimpleImputer(strategy=imputacao["numericas"])),
        ("padronizar", StandardScaler()),
    ])
    categoricas = Pipeline([
        ("imputar", SimpleImputer(strategy=imputacao["categoricas"])),
        ("one_hot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    binarias = SimpleImputer(strategy="constant", fill_value=imputacao["binarias_valor"])

    return ColumnTransformer([
        ("num_log", numericas_log, NUMERICAS_LOG),
        ("num", numericas, NUMERICAS_PADRAO),
        ("cat", categoricas, CATEGORICAS),
        ("bin", binarias, BINARIAS),
    ])

In [5]:
# demonstração: aprende com o treino e transforma o treino
preprocessador = criar_preprocessador()
X_treino_transformado = preprocessador.fit_transform(X_treino)

nomes_saida = preprocessador.get_feature_names_out()
print(f"Colunas depois do pré-processamento: {X_treino_transformado.shape[1]}")
print(nomes_saida[:8])

# conferências: nenhum vazio sobrou e as numéricas ficaram padronizadas
assert not np.isnan(X_treino_transformado).any()
print(f"Média das numéricas: {X_treino_transformado[:, :21].mean():.3f}")

Colunas depois do pré-processamento: 52
['num_log__mun_total_escolas' 'num_log__mun_populacao'
 'num_log__mun_densidade' 'num_log__mun_pct_indigena'
 'num_log__mun_pct_quilombola' 'num_log__mun_pib_per_capita'
 'num__mun_taxa_ano_anterior' 'num__mun_alunos_por_docente']
Média das numéricas: 0.000


## 5. Baselines

Dois modelos de referência, avaliados com **validação cruzada de 5 rodadas agrupada por
município** (`GroupKFold`) dentro do treino. Em cada rodada, cerca de 20% dos municípios
do treino ficam de fora para validação, e o pipeline completo (pré-processamento e
modelo) é treinado de novo só com os demais. O resultado é a média e o desvio padrão
das 5 rodadas.

| Modelo | Papel |
|---|---|
| Ingênuo (classe mais comum) | Piso: responde sempre "alfabetizado", sem olhar as features |
| Regressão logística | Primeiro modelo real; régua para os modelos da etapa seguinte |

**Decisões:**
- A regressão logística usa `class_weight="balanced"`, que dá mais peso à classe de risco
  no treino. A métrica prioritária é o recall dessa classe (notebook 02, seção 3), e sem
  o balanceamento o modelo tende a favorecer a classe majoritária.
- O `peso_aluno` não é usado nesta etapa: o baseline trata todos os alunos com o mesmo
  peso. O uso como `sample_weight` fica como ponto em aberto para a etapa de modelagem.
- As métricas de precisão, recall e F1 se referem à **classe 1 (não alfabetizado)**.

In [6]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_validate

modelos = {
    "Ingênuo (classe mais comum)": DummyClassifier(strategy="most_frequent"),
    "Regressão logística": LogisticRegression(max_iter=1000, class_weight="balanced"),
}

metricas = {
    "roc_auc": "roc_auc",
    "recall": "recall",
    "precisao": "precision",
    "f1": "f1",
    "acuracia": "accuracy",
}

validacao = GroupKFold(n_splits=5)

resultados = {}
for nome, modelo in modelos.items():
    pipeline = Pipeline([("pre", criar_preprocessador()), ("modelo", modelo)])
    cv = cross_validate(
        pipeline, X_treino, y_treino,
        groups=grupos_treino, cv=validacao, scoring=metricas,
    )
    resultados[nome] = {m: cv[f"test_{m}"] for m in metricas}
    print(f"{nome}: concluído")

tabela_cv = pd.DataFrame({
    nome: {m: f"{v.mean():.3f} ± {v.std():.3f}" for m, v in res.items()}
    for nome, res in resultados.items()
}).T
tabela_cv

Ingênuo (classe mais comum): concluído
Regressão logística: concluído


,roc_auc,recall,precisao,f1,acuracia
Ingênuo (classe mais comum),0.500 ± 0.000,0.000 ± 0.000,0.000 ± 0.000,0.000 ± 0.000,0.591 ± 0.016
Regressão logística,0.661 ± 0.007,0.652 ± 0.087,0.516 ± 0.017,0.573 ± 0.038,0.607 ± 0.017


In [7]:
CAMINHO_RELATORIOS = Path("../reports/modelagem")
CAMINHO_RELATORIOS.mkdir(parents=True, exist_ok=True)

tabela_cv.to_csv(CAMINHO_RELATORIOS / "baseline_validacao_cruzada.csv", encoding="utf-8-sig")
print(f"Gravado em: {(CAMINHO_RELATORIOS / 'baseline_validacao_cruzada.csv').resolve()}")

Gravado em: C:\Users\lfc17\tech-challenge-fase3\reports\modelagem\baseline_validacao_cruzada.csv


**Interpretação:**

- **O modelo ingênuo** acerta 59,1% (a proporção de alfabetizados no treino), mas não
  identifica nenhuma criança em risco (recall 0) e não separa as classes (ROC-AUC 0,5).
- **A regressão logística supera o ingênuo onde importa.** Ela identifica **65% das
  crianças em risco** (recall 0,652), contra nenhuma do ingênuo. Dos alunos que ela
  sinaliza, **52% estão de fato em risco** (precisão 0,516), contra 41% se os alertas
  fossem dados ao acaso (a taxa de risco da base).
- **A acurácia quase não muda** (60,7% contra 59,1%), embora o comportamento dos dois
  modelos seja completamente diferente. É a confirmação prática de que a acurácia,
  sozinha, não serve para avaliar este problema (notebook 02, seção 3).
- **O ROC-AUC de 0,661 indica um poder de separação modesto.** O resultado é coerente
  com a base: quase todas as features descrevem o município, e não o aluno, então
  crianças do mesmo município recebem praticamente a mesma previsão. É um limite dos
  dados disponíveis, a registrar nas limitações do projeto.
- **O recall varia entre as rodadas** (desvio de 0,087): o desempenho depende de quais
  municípios estão na validação. A precisão e o ROC-AUC são estáveis (desvios de 0,017
  e 0,007).

**Próximos passos (etapa de modelagem):** comparar modelos de árvore (Random Forest e
Gradient Boosting), que capturam relações não lineares; ajustar o limiar de decisão
para equilibrar recall e precisão; testar o `RobustScaler` e o uso do `peso_aluno` como
peso no treino.